# 🌱 Bitki Hastalığı Modeli — Kaggle Yeniden Eğitim

**Amaç:** Mevcut modeli (mAP50 ≈ 0.89) baştan eğitmeden, confusion matrix'te karışan sınıflara **temiz veri ekleyerek** iyileştirmek. **29 sınıf korunur** (öneriler hastalığa göre farklı olduğu için birleştirme YOK).

## Çalıştırmadan önce (ÖNEMLİ)
1. **Sağ panel → Session options → Accelerator → `GPU T4 x2` veya `GPU P100`** seç.
2. **Settings → Internet → `On`** (pip ve veri indirme için; telefon doğrulaması gerekebilir).
3. Roboflow anahtarını **Add-ons → Secrets** kısmına `ROBOFLOW_API_KEY` adıyla ekle (koda yazma!).
4. **Arka planda çalıştırmak için:** üstten **Save Version → Save & Run All (Commit)**. Tarayıcıyı kapatsan bile Kaggle sunucuda 12 saate kadar çalıştırır; bitince çıktı (`best.pt`) versiyonun output'unda olur.


## 1) Kurulum ve GPU kontrolü

In [ ]:
!pip -q install ultralytics roboflow

import torch
from ultralytics import YOLO
print('Ultralytics hazir. GPU var mi:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2) Veri setini getir

**Yeni veri nasıl eklenir?** Roboflow projene, düşük başarımlı sınıflara (özellikle `Tomato leaf bacterial spot` — recall 0.49; ayrıca `Tomato Early blight`, `Corn Gray leaf spot`) **temiz, doğru etiketli** görseller ekle → **yeni bir Version oluştur** → aşağıya o versiyon numarasını yaz.

> İpucu: Roboflow'da **Augmentation'ı KAPAT** (sadece Resize/Auto-Orient kalsın). Augmentation'ı eğitimde YOLO yapacak; diske gömülü augmentation daha kötü sonuç verir.

Aşağıdaki `WORKSPACE / PROJECT / VERSION` değerlerini kendi Roboflow projene göre düzenle.

In [ ]:
from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow

# --- DÜZENLE: kendi Roboflow projen ---
WORKSPACE = 'bengins-workspace-n2nmq'
PROJECT   = 'PLANTDOC-PROJE-SLUGUN'   # PlantDoc tabanlı (29 sinif) projenin slug'i
VERSION   = 1                          # yeni veri ekledigin son versiyon numarasi
# --------------------------------------

api_key = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
rf = Roboflow(api_key=api_key)
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download('yolov8')

DATA_YAML = f'{dataset.location}/data.yaml'
print('data.yaml:', DATA_YAML)

**Alternatif:** Roboflow yerine veriyi Kaggle Dataset olarak yükledi isen, üstteki hücreyi atla ve `DATA_YAML` yolunu elle ver:
```python
# DATA_YAML = '/kaggle/input/senin-dataset-adin/data.yaml'
```

## 3) Eğitim

Reçete: `yolo11m` (aynı boyutta yolov8m'den biraz daha iyi), 150 epoch, AdamW + cosine LR, on-the-fly augmentation. `patience=30` iyileşme durunca erken durdurur (Kaggle kotasını korur). Checkpoint `/kaggle/working` altına yazılır — oturum koparsa bile output'ta kalır.

In [ ]:
model = YOLO('yolo11m.pt')   # daha hafif/hizli istersen: 'yolo11s.pt' veya 'yolov8s.pt'

results = model.train(
    data=DATA_YAML,
    epochs=150,
    imgsz=640,
    batch=16,            # P100/T4 icin uygun; bellek tasarsa 8 yap
    patience=30,         # 30 epoch iyilesme yoksa erken dur
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,         # cosine LR — sona dogru daha stabil
    # On-the-fly augmentation (diske gomulu yerine):
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10, translate=0.1, scale=0.5,
    fliplr=0.5, mosaic=1.0, mixup=0.1,
    close_mosaic=10,     # son 10 epoch mosaic kapali — ince ayar
    seed=42,             # tekrarlanabilirlik
    project='/kaggle/working/egitim',
    name='plantdoc_v2',
)

## 4) Değerlendir — yeni model gerçekten iyileşti mi?

Confusion matrix + sınıf-bazlı mAP üretir. Eski modelin skoruyla (mAP50 **0.89**, mAP50-95 **0.65**) karşılaştır. Özellikle `Tomato leaf bacterial spot` recall'unun 0.49'dan yükselmesini bekliyoruz.

In [ ]:
best = YOLO(results.save_dir + '/weights/best.pt')

try:
    metrics = best.val(data=DATA_YAML, split='test', plots=True)
except Exception as e:
    print('test split yok, val kullaniliyor:', e)
    metrics = best.val(data=DATA_YAML, split='val', plots=True)

print('\n===== YENI MODEL =====')
print('mAP50   :', round(metrics.box.map50, 4))
print('mAP50-95:', round(metrics.box.map, 4))
print('\nSinif bazli mAP50 (dusuk = hala problemli):')
for idx, ap in sorted(zip(metrics.box.ap_class_index, metrics.box.ap50), key=lambda x: x[1]):
    print(f'  {ap:6.3f}  {best.names[int(idx)]}')

In [ ]:
# Confusion matrix'i notebook icinde goster
from IPython.display import Image as IPImage
IPImage(filename=str(metrics.save_dir) + '/confusion_matrix_normalized.png')

## 5) Modeli indir

Eğitim bitince `best.pt`'yi bilgisayarına indir:
- **Save & Run All (Commit) ile çalıştırdıysan:** notebook versiyonunun **Output** sekmesinden `best.pt`'yi indir.
- **İnteraktif çalıştırdıysan:** aşağıdaki hücre dosyayı `/kaggle/working` köküne kopyalar; sağ paneldeki **Output**'tan indirebilirsin.

Sonra `app.py` içindeki `load_model()` fonksiyonunda `YOLO("plantdoc_150epoch.pt")` satırını yeni dosya adıyla değiştir. Eski model yedekte kalır; beğenmezsen geri dönersin.

In [ ]:
import shutil
src = results.save_dir + '/weights/best.pt'
dst = '/kaggle/working/plantdoc_v2_best.pt'
shutil.copy(src, dst)
print('Indirmeye hazir:', dst)